# Python for Neuroscience (2026)

# 05. Two-photon calcium imaging: pre-processing

<!-- <img src="Images/2P_ca_invivo.png" width="800"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/2P_ca_invivo.png" width="800">

**Figure**. Representative steps of an in vivo two-photon calcium imaging experiment in mice.

After all the effort put into injecting the virus, doing the craniotomy, and recording the activity during your experiment, it is time for analysis! But first, some pre-processing is usually required. Pre-processing helps clean up the data and make it easier to extract meaningful signals from your recordings. Even simple steps like motion correction or filtering can make a big difference in the quality of your analysis.

<!-- <img src="Images/2P_ca_analysis_pipeline.png" width="1000"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/2P_ca_analysis_pipeline.png" width="1000">

## File type

It is likely that you have used a commercial 2P microscope with specific files such as "nd2" (Nikon) or "lif" (Leica). In that case, it is usually better to convert these files to standard TIFF files, unless you use libraries that can handle them directly. Some libraries to read proprietary files in Python:
- Nikon files: https://github.com/tlambert03/nd2
- Leica files: `readlif` from [AICS AICSImageIO](https://allencellmodeling.github.io/aicsimageio/)

Original files usually contain metadata that you can also export, but be aware that the microscope software may not record an accurate timestamp of the images due to software jitter and other delays.

## Motion correction
Brain motion on the order of several microns is very common in in vivo two-photon recordings, especially due to movements.d heartbeat. These small shifts can distort the images, but they can be corrected after acquisition using image registration algorithms. Mainly, rigid and non-rigid registration.
- In rigid registration, each frame is shifted as a whole in the x and y directions to align it to a reference image. A template is computed (for example, by averaging multiple frames), and each new frame is aligned to it. The displacement is estimated by computing the cross-correlation between the frame and the template. In practice, this is done using the Fast Fourier Transform (FFT).
- In non-rigid registration, the image is divided into smaller patches, and each patch is allowed to move independently. A widely used algorithm for non-rigid motion correction is NoRMCorre, created by [Pnevmatikakis and Giovannucci, 2017](https://www.sciencedirect.com/science/article/pii/S0165027017302753).


## Filtering
Filtering can help reduce noise in your recordings and improve signal quality.

## Downsampling
For some coarse analyses, you may not need the original resolution and can downsample the original stacks to speed up processing.


**GOAL**: This notebook will show several steps for motion correction of two-photon calcium imaging recordings using [pyStackReg](https://pystackreg.readthedocs.io/en/v0.2.2/index.html), and two examples for filtering and downsampling the stacks.

Most common Python libraries for the analysis of two-photon imaging data include algorithms for motion correction:
- [Suite2P](https://github.com/MouseLand/suite2p).
- [Caiman](https://github.com/flatironinstitute/CaImAn).


# Import the libraries

**NOTE**. Another useful library to load and save TIFF images is: [tifffile](https://github.com/cgohlke/tifffile)

Download [ImageJ](https://imagej.net/ij/download.html) to open the TIFF stacks.

In [ ]:
!pip install pystackreg

In [ ]:
# Import the packages
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Skimage
from skimage import io
from skimage.transform import resize

# pystackreg
from pystackreg import StackReg

# Scipy
from scipy.stats import ks_2samp
from scipy.ndimage import median_filter

# Create the paths

**Data management: Resources**
- [FAIR Guiding Principles for scientific data management and stewardship](https://www.go-fair.org/fair-principles/)
- [NIN guidelines for data storage](https://nin.nl/wp-content/uploads/sites/2/2024/10/data-storage-protocol-NIN_20211210.pdf).
- [NIH data management](https://grants.nih.gov/policy-and-compliance/policy-topics/sharing-policies/dms/data-management).
- [Cambridge research data management](https://www.data.cam.ac.uk/about).

In [ ]:
# Create a folder name for the patch-clamp notebooks
notebook_name = '2P_calcium'

# Current working directory (default is '/content' in Colab)
data_path = os.getcwd()

# Change the folder names accordingly
paths = {'data':  f'{data_path}/Example Data',
         'processed_data': f'{data_path}/Processed_data/{notebook_name}',
         'analysis': f'{data_path}/Analysis/{notebook_name}'}

# Make folders if they do not exist yet
for path in paths.values():
    os.makedirs(path, exist_ok=True)

# Download the data

In [ ]:
if 'google.colab' in str(get_ipython()):

    !wget -O "Example Data/v1_ca_01.tif" \
    "https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Example%20data/v1_ca_01.tif"

    !wget -O "Example Data/v1_ca_dendrite.tif" \
    "https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Example%20data/v1_ca_dendrite.tif"
else:
    print("Download data from 'Example Data' folder.")

# Load the data

Example data to show how to pre-process calcium imaging recordings:


<!-- <img src="Images/05_v1_ca_dendrite.png" width="300"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/05_v1_ca_dendrite.png" width="300">

- **v1_ca_01.tif**: In vivo two-photon calcium imaging of neurons from the visual cortex using the calcium sensor [GCaMP8m](https://www.nature.com/articles/s41586-023-05828-9).


<!-- <img src="Images/05_v1_ca_01.png" width="300"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/05_v1_ca_01.png" width="300">

- **v1_ca_dendrite.tif**: In vitro confocal recording of a pyramidal neuron dendrite in the visual cortex using the calcium sensor [GCaMP6s](https://www.nature.com/articles/nature12354).

In [ ]:
# Filename
experiment = 'v1_ca'
recording = '_dendrite'

# Call this variable stack if you do not need to overwrite it
stack_path = f"{paths['data']}/{experiment}{recording}.tif"

if os.path.exists(stack_path):
    stack = io.imread(stack_path)

    # Stack shape
    print("time, x, y dimensions:", stack.shape)

# Stack properties

Once you load the stack with skimage, you can treat it as a numpy array, and explore some of the properties, slicing, etc.

In [ ]:
type(stack)

In [ ]:
# Data type
print("Data type:", stack.dtype)

# Minimum and maximum intensity
print("Min intensity:", stack.min())
print("Max intensity:", stack.max())

# Mean and standard deviation
print("Mean intensity:", stack.mean())
print("Std intensity:", stack.std())

# Number of dimensions
print("Number of dimensions:", stack.ndim)

# Plot an image of the stack

- [plt.imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) for images.

In [ ]:
# Show a single frame
plt.imshow(stack[0], cmap='gray')
plt.title("First frame")
plt.show()

## Q: How to plot the average image of the time-series?

**Tip**. Use the option axis in np.mean: https://numpy.org/doc/stable/reference/generated/numpy.mean.html

In [ ]:
# Show the average image of the time-series
# plt.imshow(stack.mean(MISSING PARAMETER), cmap='gray')
plt.title("Average")
plt.show()

<details>
<summary><strong>Solution</strong></summary>

```python





# Show the average image of the time-series
plt.imshow(stack.mean(axis=0), cmap='gray')
plt.title("Average")
plt.show()
```

In [ ]:
# Show the average image of the time-series
plt.imshow(stack.mean(axis=0), cmap='gray')
plt.title("Average")
plt.show()

# Motion correction with pyStackReg

**pyStackReg** is used to align (register) one or more images to a common reference image. It is directly ported from the source code of the ImageJ plugins TurboReg and StackReg by Philippe Thévenaz (EPFL).

Motion correction of imaging data and [pyStackReg documentation](https://pystackreg.readthedocs.io/en/latest/readme.html)

pyStackReg enables different transformations, including:
- Translation. It shifts the image in x and y directions.
- Rigid body (translation + rotation). Shifting + rotating the image.
- Scaled roration. Shits + rotates + scale the image.
- Affine (Translation + Rotation + Scaling + Shearing). Preserves lines and parallelism, but not necessarily Euclidean distances and angles.
- Bilinear. It allows local distortions. This can introduce small artifacts.

## Function: correction

- `matrix = register_stack(...)`. This computes a transformation matrix for each frame that aligns the stack to a chosen reference image (but does not yet apply the alignment). See [register_stack](https://pystackreg.readthedocs.io/en/v0.2.2/api.html)
- `transformed = transform_stack(stack, tmats=matrix)`. This applies the previously computed transformations to the stack, producing a drift‑corrected stack.
- Read the [pystackreg tutorial](https://pystackreg.readthedocs.io/en/latest/tutorial.html) for more information.

In [ ]:
def drift_correction(stack, reference='mean', n_frames=100,
                     moving_average=100, method='TRANSLATION'):
    """
    Performs drift correction using StackReg.
    Five transformations are allowed:
    TRANSLATION, RIGID_BODY, SCALED_ROTATION, AFFINE, BILINEAR

    Parameters:
        stack: image stack to be registered (time, x, y)
        reference: register to 'mean' or 'first' image
        n_frames: number of frames used to compute reference (if reference='first')
        moving_average: frames used in moving average during registration
        method: transformation model (default 'TRANSLATION')

    Returns:
        stack_corrected, transformation matrix

    Note: matrix shape is (n_frames, 3, 3)
          [frame, 0, 2] and [frame, 1, 2] are X and Y shifts
    """

    # Map method string to StackReg constant
    method_dict = {
        'TRANSLATION': StackReg.TRANSLATION,
        'RIGID_BODY': StackReg.RIGID_BODY,
        'SCALED_ROTATION': StackReg.SCALED_ROTATION,
        'AFFINE': StackReg.AFFINE,
        'BILINEAR': StackReg.BILINEAR}

    stack_registrator = StackReg(method_dict[method.upper()])

    # Compute transformation matrices
    transformation_matrices = stack_registrator.register_stack(
        stack, reference=reference, n_frames=n_frames, moving_average=moving_average)

    # Apply transformations
    stack_corrected = stack_registrator.transform_stack(stack, tmats=transformation_matrices)

    return stack_corrected, transformation_matrices

## Run the function

**Tip**: `%%time` is Jupyter lab magic command to print the running time for the entire cell

In [ ]:
%%time
stack_mc, matrix = drift_correction(stack, moving_average=100)

## Save the file

In [ ]:
processed_file_path = f"{paths['processed_data']}/{experiment}{recording}_mc.tif"
io.imsave(processed_file_path, stack_mc, check_contrast=False)
print(processed_file_path)

## Q: Check the shape and type object of stack and stack_mc

**Tip**: Use `.shape` and `.dtype`. Do the stacks have the same properties?

In [ ]:
# Compare original and corrected stack


<details>
<summary><strong>Solution</strong></summary>

```python

# Compare original and corrected stack
print("Original stack shape:", stack.shape, "dtype:", stack.dtype)
print("Corrected stack shape:", stack_mc.shape, "dtype:", stack_mc.dtype)
```

In [ ]:
# Compare original and corrected stack
print("Original stack shape:", stack.shape, "dtype:", stack.dtype)
print("Corrected stack shape:", stack_mc.shape, "dtype:", stack_mc.dtype)

## Function: correct dtype

In [ ]:
def drift_correction(stack, reference='mean', n_frames=100,
                     moving_average=100, method='TRANSLATION'):
    """
    Performs drift correction using StackReg.
    Five transformations are allowed:
    TRANSLATION, RIGID_BODY, SCALED_ROTATION, AFFINE, BILINEAR

    Parameters:
        stack: image stack to be registered (time, x, y)
        reference: register to 'mean' or 'first' image
        n_frames: number of frames used to compute reference (if 'first')
        moving_average: frames used in moving average during registration
        method: transformation model (default 'TRANSLATION')

    Returns:
        stack_corrected, transformation matrix

    Note: matrix shape is (n_frames, 3, 3)
          [frame, 0, 2] and [frame, 1, 2] are X and Y shifts
    """

    # Map method string to StackReg constant
    method_dict = {
        'TRANSLATION': StackReg.TRANSLATION,
        'RIGID_BODY': StackReg.RIGID_BODY,
        'SCALED_ROTATION': StackReg.SCALED_ROTATION,
        'AFFINE': StackReg.AFFINE,
        'BILINEAR': StackReg.BILINEAR}

    stack_registrator = StackReg(method_dict[method.upper()])

    # Compute transformation matrices
    transformation_matrices = stack_registrator.register_stack(
        stack, reference=reference, n_frames=n_frames, moving_average=moving_average)

    # Apply transformations
    stack_corrected = stack_registrator.transform_stack(stack, tmats=transformation_matrices)

    # RETURN SAME DTYPE AS ORIGINAL STACK
    stack_corrected = np.round(stack_corrected).astype(stack.dtype)

    return stack_corrected, transformation_matrices

## Check again the function and stack_mc dtype

In [ ]:
%%time
stack_mc, matrix = drift_correction(stack, method='RIGID_BODY', moving_average=50)

# Print shape
# COMPLETE CODE

<details>
<summary><strong>Solution</strong></summary>

```python

# Compare original and corrected stack
print("Original stack shape:", stack.shape, "dtype:", stack.dtype)
print("Corrected stack shape:", stack_mc.shape, "dtype:", stack_mc.dtype)
```

In [ ]:
# Compare original and corrected stack
print("Original stack shape:", stack.shape, "dtype:", stack.dtype)
print("Corrected stack shape:", stack_mc.shape, "dtype:", stack_mc.dtype)

In [ ]:
processed_file_path = f"{paths['processed_data']}/{experiment}{recording}_mc.tif"
io.imsave(processed_file_path, stack_mc, check_contrast=False)
print(processed_file_path)

In [ ]:
%%time
stack_mc, matrix = drift_correction(stack, moving_average=100)

# Compare original and corrected stack
print("Original stack shape:", stack.shape, "dtype:", stack.dtype)
print("Corrected stack shape:", stack_mc.shape, "dtype:", stack_mc.dtype)

<details>
<summary><strong>Solution</strong></summary>

```python

%%time
stack_mc, matrix = drift_correction(stack, moving_average=100)

# Compare original and corrected stack
print("Original stack shape:", stack.shape, "dtype:", stack.dtype)
print("Corrected stack shape:", stack_mc.shape, "dtype:", stack_mc.dtype)

```

## Q: Plot the average image of stack and stack_mc

**Tip**:
- Use `ax.imshow`
- Optional: adjust the contrast using imshow parameters: `vmin` and `vmax`. For example, vmin = stack.mean(axis=0).min()
vmax = stack.mean(axis=0).max()

In [ ]:
# Plot
# Missing line fig, axes

# Set vmin and vmax for automatic contrast
vmin_stack, vmax_stack = # COMPLETE CODE
vmin_stack_mc, vmax_stack_mc = # COMPLETE CODE

# Average of original stack
# COMPLETE: im.show(# stack_mc + vmin, vmax)
axes[0].set_title("Original stack average")
axes[0].axis('off')

# Average of corrected stack
# COMPLETE: im.show(# stack_mc + vmin, vmax)
axes[1].set_title("Corrected stack average")
axes[1].axis('off')

plt.show()


<details>
<summary><strong>Solution</strong></summary>

```python


# Plot
fig, axes = plt.subplots(2, 1, figsize=(6, 6))

# Set vmin and vmax for automatic contrast
vmin_stack, vmax_stack = stack.mean(axis=0).min(), stack.mean(axis=0).max()
vmin_stack_mc, vmax_stack_mc = stack_mc.mean(axis=0).min(), stack_mc.mean(axis=0).max()

# Average of original stack
axes[0].imshow(stack.mean(axis=0), cmap='gray', vmin=vmin_stack, vmax=vmax_stack)
axes[0].set_title("Original stack average")
axes[0].axis('off')

# Average of corrected stack
axes[1].imshow(stack_mc.mean(axis=0), cmap='gray', vmin=vmin_stack_mc, vmax=vmax_stack_mc)
axes[1].set_title("Corrected stack average")
axes[1].axis('off')

plt.show()

```

In [ ]:

# Plot
fig, axes = plt.subplots(2, 1, figsize=(6, 6))

# Set vmin and vmax for automatic contrast
vmin_stack, vmax_stack = stack.mean(axis=0).min(), stack.mean(axis=0).max()
vmin_stack_mc, vmax_stack_mc = stack_mc.mean(axis=0).min(), stack_mc.mean(axis=0).max()

# Average of original stack
axes[0].imshow(stack.mean(axis=0), cmap='gray')
axes[0].set_title("Original stack average")
axes[0].axis('off')

# Average of corrected stack
axes[1].imshow(stack_mc.mean(axis=0), cmap='gray')
axes[1].set_title("Corrected stack average")
axes[1].axis('off')

plt.show()

## Function: add drifting plot

In [ ]:
def drift_correction(stack, reference='mean', n_frames=100,
                     moving_average=100, method='TRANSLATION'):
    """
    Performs drift correction using StackReg.
    Five transformations are allowed:
    TRANSLATION, RIGID_BODY, SCALED_ROTATION, AFFINE, BILINEAR

    Parameters:
        stack: image stack to be registered (time, x, y)
        reference: register to 'mean' or 'first' image
        n_frames: number of frames used to compute reference (if 'first')
        moving_average: frames used in moving average during registration
        method: transformation model (default 'TRANSLATION')

    Returns:
        stack_corrected, transformation matrix

    Note: matrix shape is (n_frames, 3, 3)
          [frame, 0, 2] and [frame, 1, 2] are X and Y shifts
    """

    # Map method string to StackReg constant
    method_dict = {
        'TRANSLATION': StackReg.TRANSLATION,  # x–y translation + small rotation
        'RIGID_BODY': StackReg.RIGID_BODY,
        'SCALED_ROTATION': StackReg.SCALED_ROTATION,
        'AFFINE': StackReg.AFFINE,
        'BILINEAR': StackReg.BILINEAR}

    stack_registrator = StackReg(method_dict[method.upper()])

    # Compute transformation matrices
    transformation_matrices = stack_registrator.register_stack(
        stack, reference=reference, n_frames=n_frames, moving_average=moving_average)

    # Apply transformations
    stack_corrected = stack_registrator.transform_stack(stack, tmats=transformation_matrices)

    # NEW: Compute drift (dx, dy)
    dx, dy = [], []
    for frame in transformation_matrices:
        dx.append(frame[0, 2])
        dy.append(frame[1, 2])

    # Plot
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(6, 10))

    # Average of original stack
    ax1.imshow(stack.mean(axis=0), cmap='gray')
    ax1.set_title("Original stack average")
    ax1.axis('off')

    # Average of corrected stack
    ax2.imshow(stack_corrected.mean(axis=0), cmap='gray')
    ax2.set_title("Corrected stack average")
    ax2.axis('off')

    # Drift over frames
    ax3.plot(dx, label='x')
    ax3.plot(dy, label='y')
    ax3.set_title("Drift")
    ax3.set_xlabel("Frame")
    ax3.set_ylabel("Pixel shift")
    ax3.legend()

    plt.tight_layout()
    plt.show()

    return stack_corrected, transformation_matrices

In [ ]:
%%time
stack_mc, matrix = drift_correction(stack, reference='mean', method='TRANSLATION', moving_average=100)

# Save the motion corrected stack
processed_file_path = f"{paths['processed_data']}/{experiment}{recording}_mc.tif"
io.imsave(processed_file_path, stack_mc, check_contrast=False)
print(processed_file_path)

## Compare drift correction methods

In [ ]:
# List of all StackReg methods
methods = ['AFFINE', 'RIGID_BODY']

# Dictionary to store results
results = {}

for method in methods:
    print(f"{method}")

    corrected_stack, matrices = drift_correction(stack, method=method)

    # Store corrected stack and drift for comparison
    results[method] = {
        'stack': corrected_stack,
        'dx': [frame[0,2] for frame in matrices],
        'dy': [frame[1,2] for frame in matrices]}

In [ ]:
# TRY THE TWO-PHOTON CALCIUM IMAGING STACK
stack_dendrite = io.imread(f"{paths['data']}/v1_ca_01.tif")

# List of all StackReg methods
methods = ['AFFINE', 'RIGID_BODY']

# Dictionary to store results
results = {}

for method in methods:
    print(f"{method}")

    corrected_stack_dendrite, matrices = drift_correction(stack_dendrite, method=method)

    # Store corrected stack and drift for comparison
    results[method] = {
        'stack': corrected_stack_dendrite,
        'dx': [frame[0,2] for frame in matrices],
        'dy': [frame[1,2] for frame in matrices]}

# Cumulative intensity comparison

This plot compares the cumulative histogram of pixel intensities before and after motion correction.
If the curves overlap closely, the registration preserved the global intensity distribution (i.e., no major intensity distortions were introduced).

Register the stack to `mean` seems to work better than to the first `n_frames`. Using `translation` and a `rolling average` between 50-100, it smooths the correction but it still creates some artifacts. Other transformations do not seem to improve the correction but play a bit with some of your files. You can compare the results with the motion correction function included in the `Caiman` library.

In [ ]:
fig, ax = plt.subplots()

ax.hist(stack.flatten(), density= True, bins = 300, cumulative=True,
        histtype='step', fill=False, edgecolor='gray', label='original')

ax.hist(stack_mc.flatten(), density= True, bins = 300, cumulative=True,
        histtype='step', fill=False, edgecolor='magenta', label='mc')

ax.set_ylabel('Normalized count')
ax.set_xlabel('Pixel value')
ax.legend()

fig.savefig(f"{paths['analysis']}/{experiment}{recording}_histo.png")

plt.show()

## Q: Compare the distribution values

**Tips**:
- density=False and Kolmogorov-Smirnov: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ks_2samp.html
- Sanity check: [np.histogram](https://numpy.org/doc/stable/reference/generated/numpy.histogram.html) and [np.cumsum]( https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html)

In [ ]:
bins = 300

# Flatten stacks
stack_px = stack.flatten()
stack_mc_px = stack_mc.flatten()

# Plot cumulative histogram (density=False)
fig, ax = plt.subplots(figsize=(6,4))

# MISSING LINE
ax.hist(stack.flatten(), density= False, bins = 300, cumulative=True,
        histtype='step', fill=False, edgecolor='gray', label='original')

# MISSING LINE
ax.hist(stack_mc.flatten(), density= False, bins = 300, cumulative=True,
        histtype='step', fill=False, edgecolor='magenta', label='mc')

ax.set_ylabel('Cumulative count')
ax.set_xlabel('Pixel value')
ax.set_title('Cumulative intensity histogram')
ax.legend()
plt.show()

# Kolmogorov-Smirnov test on raw pixel values
# MISSING LINE
ks_stat, p_value = ks_2samp(stack_px, stack_mc_px)
print(f"KS statistic: {ks_stat:.4f}")
print(f"p-value: {p_value:.4f}")

# Print total pixel counts (sanity check)
# counts_orig, bin_edges =
# counts_mc, _ = np.histogram(stack_mc_px, bins=bin_edges)
counts_orig, bin_edges = np.histogram(stack_px, bins=bins)
counts_mc, _ = np.histogram(stack_mc_px, bins=bin_edges)

# Compute cumulative counts
# cum_counts_orig =
# cum_counts_mc =
cum_counts_orig = np.cumsum(counts_orig)
cum_counts_mc = np.cumsum(counts_mc)

print(f"Total pixel count (original): {np.sum(counts_orig)}")
print(f"Total pixel count (motion-corrected): {np.sum(counts_mc)}")

<details>
<summary><strong>Solution</strong></summary>

```python

bins = 300

# Flatten stacks
stack_px = stack.flatten()
stack_mc_px = stack_mc.flatten()

# Plot cumulative histogram using step
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(stack.flatten(), density= False, bins = 300, cumulative=True,
        histtype='step', fill=False, edgecolor='gray', label='original')

ax.hist(stack_mc.flatten(), density= False, bins = 300, cumulative=True,
        histtype='step', fill=False, edgecolor='magenta', label='mc')

ax.set_ylabel('Cumulative count')
ax.set_xlabel('Pixel value')
ax.set_title('Cumulative intensity histogram')
ax.legend()
plt.show()

# Kolmogorov-Smirnov test on raw pixel values
ks_stat, p_value = ks_2samp(stack_px, stack_mc_px)
print(f"KS statistic: {ks_stat:.4f}")
print(f"p-value: {p_value:.4f}")

# Print total pixel counts (sanity check)
counts_orig, bin_edges = np.histogram(stack_px, bins=bins)
counts_mc, _ = np.histogram(stack_mc_px, bins=bin_edges)

# Compute cumulative counts
cum_counts_orig = np.cumsum(counts_orig)
cum_counts_mc = np.cumsum(counts_mc)

print(f"Total pixel count (original): {np.sum(counts_orig)}")
print(f"Total pixel count (motion-corrected): {np.sum(counts_mc)}")

```

# Filter the stack

To smooth edges and remove outliers, it is sometimes useful to apply a filter to the image stack. The scipy.ndimage module contains several functions for multidimensional image processing, including 2D and 3D filtering. The [median.filter](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.median_filter.html#scipy.ndimage.median_filter) is very common in confocal and two-photon imaging because it reduces noise while preserving edges.

- size. Function adjusts size to the number of dimensions of the input array, so that, if the input array is shape (10,10,10), and size is 2, then the actual size used is (2,2,2).

In [ ]:
# Example: median filter with a radius of 1 pixel
radius_px = 1

# Convert radius to window size (2 * radius + 1)
size_xy = 2 * radius_px + 1  # 3

# Apply median filter only in x and y (not across frames)
stack_mc_filtered = median_filter(stack_mc, size=(1, size_xy, size_xy))

# Save filtered stack
processed_file_path = f"{paths['processed_data']}/{experiment}{recording}_mc_filtered.tif"
io.imsave(processed_file_path, stack_mc, check_contrast=False)
print(processed_file_path)

## Q: Plot average image of stack_mc and stack_mc_filtered

**Tips**:
- Use im.show()

In [ ]:
# Plot
fig, axes = plt.subplots(2, 1, figsize=(10, 10))

# Average of original stack
# MISSING LINE
# Add title (optional)

# Average of corrected stack
# MISSING LINE
# Add title (optional)

plt.show()

In [ ]:
# Plot
fig, axes = plt.subplots(2, 1, figsize=(10, 10))

# Average of original stack
axes[0].imshow(stack_mc.mean(axis=0), cmap='gray')
axes[0].set_title("Stack_mc")

# Average of corrected stack
axes[1].imshow(stack_mc_filtered.mean(axis=0), cmap='gray')
axes[1].set_title("Stack_mc_filtered")

plt.show()

<details>
<summary><strong>Solution</strong></summary>

```python

# Plot
fig, axes = plt.subplots(2, 1, figsize=(10, 10))

# Average of original stack
axes[0].imshow(stack_mc.mean(axis=0), cmap='gray')
axes[0].set_title("Stack_mc")

# Average of corrected stack
axes[1].imshow(stack_mc_filtered.mean(axis=0), cmap='gray')
axes[1].set_title("Stack_mc_filtered")

plt.show()

```

# Downsampling

Downsampling using the [skimage.transform.resize](https://scikit-image.org/docs/0.25.x/api/skimage.transform.html#skimage.transform.resize).

As previously, you can plot cumulative histograms to compare the pixel distribution before and after downsamplingTo identify artifacts and possible changes in image intensity values after downsampling.

In [ ]:
def downsample_stack(stack, downsample_factor, dtype=np.uint16):
    """
    Downsamples a stack in X and Y by a given factor.

    Parameters:
        stack: NumPy array of shape (t, x, y)
        downsample_factor: factor to reduce X and Y dimensions
        dtype: data type of the returned stack (default: np.uint16)

    Returns:
        downsampled_stack: stack with smaller X and Y dimensions
    """
    t, x, y = stack.shape
    new_shape = (t, x // downsample_factor, y // downsample_factor)

    downsampled_stack = resize(stack, new_shape,
                               order=0,  # nearest neighbor
                               preserve_range=True,
                               anti_aliasing=True).astype(dtype)
    return downsampled_stack

## Q: Apply the function and save the resized stack with another name

In [ ]:
# Downsample stack_mc by a factor of 2
# stack_mc_ds = # COMPLETE THE CODE (apply the function)

# Print shape
# COMPLETE THE CODE

# Save the mc_stack_ds
# file_path = # COMPLETE THE CODE
# io.imsave(# COMPLETE THE CODE)

In [ ]:
# Downsample stack_mc by a factor of 2
stack_mc_ds = downsample_stack(stack_mc, downsample_factor=2)

# Print shape
print(stack_mc_ds.shape)

# Save the mc_stack_ds
file_path = f"{paths['processed_data']}/{experiment}{recording}_mc_ds.tif"
io.imsave(file_path, stack_mc_ds, check_contrast=False)
print(processed_file_path)


<details>
<summary><strong>Solution</strong></summary>

```python

# Downsample stack_mc by a factor of 2
stack_mc_ds = downsample_stack(stack_mc, downsample_factor=2)

# Print shape
print(stack_mc_ds.shape)

# Save the mc_stack_ds
file_path = f"{paths['processed_data']}/{experiment}{recording}_mc_ds.tif"
io.imsave(file_path, stack_mc_ds, check_contrast=False)
print(processed_file_path)

```